# 🤖 Grant Agent — Google Colab Runner

Runs the **complete Grant Agent platform** (FastAPI + Playwright) inside Colab, with:
- 📁 **Google Drive** persistence — database, uploads, screenshots all survive session resets
- 🌐 **cloudflared** public URL so you can access the API from any browser or phone
- 🔄 **Keep-alive pings** every 30 s to maximise uptime
- 🤖 **Real-site browser automation** — Playwright navigates real grant portals and fills your email

### ⚠️ Colab Session Limits
| Plan | Max session | Notes |
|------|-------------|-------|
| Free | ~12 h | Disconnects if browser tab is closed |
| Pro  | ~24 h | Background execution available |

All data is on **Google Drive** — just re-run after a restart and you pick up exactly where you left off.

---

## 📋 Run Order
1. **Cell 1** — Mount Drive, install deps (≈5 min first time)
2. **Cell 2** — Edit your email & Gemini API key
3. **Cell 3** — Launch server + tunnel (keep this cell running)
4. *(Optional)* **Cell 4** — Write your email to the org profile so the bot uses it
5. *(Optional)* **Cell 5** — Test the browser bot on a real grant site
6. *(Optional)* **Cell 6** — Discover/scrape a new grant from any URL

In [ ]:
# ============================================================
# CELL 1 — Mount Google Drive & Install Dependencies
# First run takes ~5 minutes. Subsequent runs: ~30 seconds.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

# ── Persistent directories on Drive ──────────────────────────
DATA_DIR = '/content/drive/MyDrive/GrantAgentData'
for sub in ['db', 'uploads', 'screenshots', 'logs']:
    os.makedirs(f'{DATA_DIR}/{sub}', exist_ok=True)
print('✅ Drive directories ready:', DATA_DIR)

# ── Clone or update repository ───────────────────────────────
REPO_DIR = '/content/grant-agent'
REPO_URL = 'https://github.com/ademikelz19-hub/cookie-project.git'
BRANCH   = 'feature/grant-agent'

if os.path.exists(f'{REPO_DIR}/.git'):
    print('🔄 Pulling latest changes...')
    subprocess.run(['git', 'pull', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    print('📥 Cloning repository...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

# ── Install Python dependencies ───────────────────────────────
print('📦 Installing Python dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                f'{REPO_DIR}/grant-agent-api/requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                f'{REPO_DIR}/grant-agent-worker/requirements.txt'], check=True)

# ── Install Playwright Chromium ───────────────────────────────
print('🎭 Installing Playwright Chromium...')
subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium', '--with-deps'], check=True)

# ── Install cloudflared ───────────────────────────────────────
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('🌐 Installing cloudflared...')
    subprocess.run([
        'bash', '-c',
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
        '-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared'
    ], check=True)

print('\n✅ All dependencies installed!')

In [ ]:
# ============================================================
# CELL 2 — Configure Your Settings
# EDIT the values below, then run this cell.
# ============================================================

# ── Your Gemini API key ───────────────────────────────────────
# Free key at: https://aistudio.google.com/app/apikey
GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY_HERE'

# ── Your real email (the browser bot fills this on grant forms) ─
YOUR_EMAIL = 'your.real.email@example.com'

# ── API port ──────────────────────────────────────────────────
API_PORT = 8000

# ── JWT secret (auto-generated, fine for Colab) ───────────────
import secrets
JWT_SECRET = secrets.token_hex(32)

print('⚙️  Configuration saved.')
print(f'   Email     : {YOUR_EMAIL}')
print(f'   Gemini    : {GEMINI_API_KEY[:8]}...' if len(GEMINI_API_KEY) > 12 else f'   Gemini    : {GEMINI_API_KEY}')
print(f'   Port      : {API_PORT}')

In [ ]:
# ============================================================
# CELL 3 — Launch Grant Agent Server + cloudflared Tunnel
# Keep this cell running the whole time you use Grant Agent.
# ============================================================

import os, subprocess, threading, time, re, sys
import urllib.request

REPO_DIR = '/content/grant-agent'
DATA_DIR = '/content/drive/MyDrive/GrantAgentData'

# ── Write .env ────────────────────────────────────────────────
env_content = f"""# Grant Agent — Colab session .env
DATABASE_URL=sqlite:///{DATA_DIR}/db/grant_agent.db
SECRET_KEY={JWT_SECRET}
ALGORITHM=HS256
ACCESS_TOKEN_EXPIRE_MINUTES=10080
GEMINI_API_KEY={GEMINI_API_KEY}
PRIMARY_GEMINI_MODEL=gemini-2.0-flash
FALLBACK_GEMINI_MODEL=gemini-1.5-flash
SECONDARY_FALLBACK_GEMINI_MODEL=gemini-1.5-flash-8b
GEMINI_MAX_RETRIES=4
GEMINI_RETRY_BASE_SECONDS=2
GEMINI_REQUEST_TIMEOUT_SECONDS=45
ENVIRONMENT=colab
PROJECT_NAME=Grant Agent
UPLOAD_DIR={DATA_DIR}/uploads
SCREENSHOTS_DIR={DATA_DIR}/screenshots
CORS_ORIGINS=["*"]
LOG_LEVEL=INFO
"""

env_path = f'{REPO_DIR}/.env'
with open(env_path, 'w') as f:
    f.write(env_content)
print('✅ .env written')

# ── PYTHONPATH ───────────────────────────────────────────────
api_path    = f'{REPO_DIR}/grant-agent-api'
worker_path = f'{REPO_DIR}/grant-agent-worker'
for p in [api_path, worker_path]:
    if p not in sys.path:
        sys.path.insert(0, p)

env = os.environ.copy()
env['PYTHONPATH'] = f'{api_path}:{worker_path}'

# ── Start FastAPI with uvicorn ────────────────────────────────
log_file = open(f'{DATA_DIR}/logs/api_server.log', 'a')
print(f'🚀 Starting Grant Agent API on port {API_PORT}...')
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app',
     '--host', '0.0.0.0', '--port', str(API_PORT), '--log-level', 'info'],
    cwd=api_path, env=env,
    stdout=log_file, stderr=subprocess.STDOUT
)

time.sleep(6)
for attempt in range(15):
    try:
        urllib.request.urlopen(f'http://localhost:{API_PORT}/health', timeout=3)
        print('✅ API server is up!')
        break
    except Exception:
        time.sleep(2)
else:
    print('❌ Server failed. Check:', f'{DATA_DIR}/logs/api_server.log')
    raise RuntimeError('Server startup failed')

# ── cloudflared tunnel ────────────────────────────────────────
tunnel_url = None
tunnel_log  = []

def read_tunnel(proc):
    global tunnel_url
    for line in iter(proc.stdout.readline, b''):
        s = line.decode('utf-8', errors='ignore').strip()
        tunnel_log.append(s)
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', s)
        if m:
            tunnel_url = m.group(0)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{API_PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
threading.Thread(target=read_tunnel, args=(tunnel_proc,), daemon=True).start()

print('⏳ Waiting for tunnel URL...')
for _ in range(30):
    if tunnel_url:
        break
    time.sleep(2)

if tunnel_url:
    print('\n' + '='*60)
    print('🌐  Grant Agent is LIVE at:')
    print(f'    {tunnel_url}')
    print(f'\n📖  API Docs : {tunnel_url}/docs')
    print(f'❤️   Health   : {tunnel_url}/health')
    print('='*60)
    print('\n🔑  Default login:')
    print('    Email    : demo@grantagent.ai')
    print('    Password : password123')
    print('\n⚠️  URL changes on every restart — run Cell 3 again to get a new one.')
else:
    print('⚠️  Tunnel URL not detected. Check tunnel_log variable.')

# ── Keep-alive (30 s ping to prevent idle timeout) ────────────
def _keep_alive():
    while True:
        try:
            urllib.request.urlopen(f'http://localhost:{API_PORT}/health', timeout=5)
        except Exception:
            pass
        time.sleep(30)

threading.Thread(target=_keep_alive, daemon=True).start()
print('\n💓 Keep-alive active (30 s pings)')

In [ ]:
# ============================================================
# CELL 4 — Write YOUR Email to the Organisation Profile
# After this, the browser bot will auto-fill YOUR_EMAIL
# whenever it detects an email field on a real grant form.
# Run AFTER Cell 3.
# ============================================================

import urllib.request, urllib.parse, json

BASE = f'http://localhost:{API_PORT}/api/v1'

# Login
login_data = urllib.parse.urlencode({'username': 'demo@grantagent.ai', 'password': 'password123'}).encode()
req = urllib.request.Request(f'{BASE}/auth/login', data=login_data)
with urllib.request.urlopen(req) as r:
    TOKEN = json.loads(r.read())['access_token']
print('✅ Logged in')

headers = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

# List organisations
req = urllib.request.Request(f'{BASE}/organisations', headers=headers)
with urllib.request.urlopen(req) as r:
    orgs = json.loads(r.read())
print(f'📋 Found {len(orgs)} organisation(s):')
for i, o in enumerate(orgs):
    print(f'   [{i}] {o["organisation_name"]}  — current email: {o.get("email", "(none)")}')

# Patch every org with YOUR_EMAIL (set in Cell 2)
for org in orgs:
    # Build a full update payload preserving existing data
    payload = {k: v for k, v in org.items() if k not in ['id', 'owner_id', 'created_at', 'updated_at',
                'founders', 'team_members', 'metrics', 'documents']}
    payload['email'] = YOUR_EMAIL
    data = json.dumps(payload).encode()
    req = urllib.request.Request(
        f'{BASE}/organisations/{org["id"]}',
        data=data, headers=headers, method='PUT'
    )
    with urllib.request.urlopen(req) as r:
        updated = json.loads(r.read())
    print(f'✅ {updated["organisation_name"]} → email: {updated["email"]}')

print(f'\n🎯 Done! The browser agent will now fill "{YOUR_EMAIL}" on all grant application forms.')

In [ ]:
# ============================================================
# CELL 5 — Browser Automation: Navigate a Real Grant Site
# The Playwright bot will open the URL, scan the form,
# fill all fields it can (including your email), and report.
# It STOPS at CAPTCHA / OTP / 2FA and shows a screenshot.
# ============================================================

import urllib.request, urllib.parse, json, time

BASE = f'http://localhost:{API_PORT}/api/v1'

# ── Change to any real grant application URL ──────────────────
# The bot will navigate there and try to fill the form.
TEST_GRANT_URL = 'https://www.tonyelumelufoundation.org/apply'  # replace with real URL
ORG_INDEX = 0   # 0 = Lioris, 1 = Princess Sara Foundation

# Login
login_data = urllib.parse.urlencode({'username': 'demo@grantagent.ai', 'password': 'password123'}).encode()
req = urllib.request.Request(f'{BASE}/auth/login', data=login_data)
with urllib.request.urlopen(req) as r:
    TOKEN = json.loads(r.read())['access_token']
headers = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

# Get org
req = urllib.request.Request(f'{BASE}/organisations', headers=headers)
with urllib.request.urlopen(req) as r:
    orgs = json.loads(r.read())
ORG_ID = orgs[ORG_INDEX]['id']
print(f'🏢 Using org: {orgs[ORG_INDEX]["organisation_name"]} (email: {orgs[ORG_INDEX].get("email")})')

# Get first grant
req = urllib.request.Request(f'{BASE}/grants', headers=headers)
with urllib.request.urlopen(req) as r:
    grants = json.loads(r.read())
if not grants:
    print('No grants found. Run Cell 6 to scrape a grant URL first.')
    raise SystemExit(0)
GRANT_ID = grants[0]['id']
print(f'📄 Using grant: {grants[0]["grant_name"]}')

# Create application
app_data = json.dumps({'org_id': ORG_ID, 'grant_id': GRANT_ID}).encode()
req = urllib.request.Request(f'{BASE}/applications', data=app_data, headers=headers)
with urllib.request.urlopen(req) as r:
    application = json.loads(r.read())
APP_ID = application['id']
print(f'📝 Application ID: {APP_ID}')

# Start browser session — target_url overrides the grant's stored URL
browser_payload = json.dumps({
    'application_id': APP_ID,
    'target_url': TEST_GRANT_URL
}).encode()
req = urllib.request.Request(
    f'{BASE}/browser/start',
    data=browser_payload, headers=headers
)
with urllib.request.urlopen(req) as r:
    session = json.loads(r.read())
SESSION_ID = session.get('id')
print(f'\n🤖 Browser session started: {SESSION_ID}')
print('⏳ Waiting for automation (up to 2 minutes)...')

# Poll for result
result = {}
for _ in range(24):
    time.sleep(5)
    req = urllib.request.Request(f'{BASE}/browser/session/{SESSION_ID}', headers=headers)
    with urllib.request.urlopen(req) as r:
        result = json.loads(r.read())
    st = result.get('status', 'unknown')
    print(f'   Status: {st}')
    if st in ['COMPLETED', 'WAITING_FOR_USER', 'FAILED', 'CANCELLED']:
        break

# Report
print('\n' + '='*60)
print('📊 BROWSER AUTOMATION RESULT')
print('='*60)
print(f'  Status         : {result.get("status")}')
print(f'  Fields filled  : {result.get("completed_fields_count", 0)}')
print(f'  Current URL    : {result.get("current_url", "n/a")}')
if result.get('intervention_required'):
    print(f'\n⚠️  HUMAN NEEDED : {result.get("current_task")}')
    print(f'   Screenshot   : {result.get("latest_screenshot_path")}')
    print('   → The screenshot is saved to your Google Drive screenshots folder.')
print('\nEvent log saved in browser session record.')

In [ ]:
# ============================================================
# CELL 6 — Discover a New Grant by Scraping Any URL
# Paste a grant program page. The AI scout extracts the
# name, funder, amount, deadline, stage, and eligibility,
# then saves it to your database for matching.
# ============================================================

import urllib.request, urllib.parse, json

BASE = f'http://localhost:{API_PORT}/api/v1'

# ── Paste your grant URL here ─────────────────────────────────
GRANT_PAGE_URL = 'https://www.tonyelumelufoundation.org/entrepreneurship'

# Login
login_data = urllib.parse.urlencode({'username': 'demo@grantagent.ai', 'password': 'password123'}).encode()
req = urllib.request.Request(f'{BASE}/auth/login', data=login_data)
with urllib.request.urlopen(req) as r:
    TOKEN = json.loads(r.read())['access_token']
headers = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

# Scrape
payload = json.dumps({'url': GRANT_PAGE_URL}).encode()
req = urllib.request.Request(f'{BASE}/grants/scrape', data=payload, headers=headers)
with urllib.request.urlopen(req) as r:
    result = json.loads(r.read())

print('='*60)
print('🔍 GRANT DISCOVERED & ADDED TO DATABASE')
print('='*60)
print(f'  Name         : {result.get("grant_name")}')
print(f'  Funder       : {result.get("funder")}')
print(f'  Amount       : {result.get("funding_amount_min")} – {result.get("funding_amount_max")} {result.get("currency")}')
print(f'  Deadline     : {result.get("deadline")}')
print(f'  Stage class  : {result.get("stage_classification")}')
print(f'  App status   : {result.get("application_status")}')
print(f'  Verification : {result.get("verification_status")}')
print(f'  Grant ID     : {result.get("id")}')
print('\nThis grant is now in your database. Run Cell 5 to apply!')